# Inverse Problem (two unknowns): recover $c$ AND $\nu$ with a PINN

PDE:  $u_t + c\,u_x = \nu\,u_{xx}$   (convection-diffusion), with **both**
the convection speed $c$ and the diffusion coefficient $\nu$ **unknown**.

We are given only sparse, noisy sensor readings of $u$. No $c$, no $\nu$, no IC/BC.
The PINN recovers **both parameters** and the full field in a single training run.

**Analytic solution used only to fake the data:** a Gaussian advects at speed $c$
and spreads by diffusion,
$$u(x,t)=\frac{\sigma_0}{\sigma(t)}\exp\!\Big(-\frac{(x-x_0-c\,t)^2}{2\,\sigma(t)^2}\Big),
\quad \sigma(t)^2=\sigma_0^2+2\nu t.$$

Runs in a few seconds on Colab (CPU or GPU).

In [ ]:
# Cell 1 -- Imports, device, hidden ground truth
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

C_TRUE   = 1.0        # <-- unknown convection speed
NU_TRUE  = 0.02       # <-- unknown diffusion coefficient
L, T     = 2.0, 1.0
X0, SIG0 = 0.4, 0.12  # initial Gaussian center / width

def exact(x, t):      # analytic convection-diffusion solution (data only)
    exp  = torch.exp  if torch.is_tensor(x) else np.exp
    sqrt = torch.sqrt if torch.is_tensor(x) else np.sqrt
    sig  = sqrt(SIG0**2 + 2*NU_TRUE*t)
    return (SIG0/sig) * exp(-((x - X0 - C_TRUE*t)**2) / (2*sig**2))

In [ ]:
# Cell 2 -- Sparse, noisy measurements (all the PINN ever sees)
N_DATA = 80
NOISE  = 0.02
xd = np.random.rand(N_DATA) * L
td = np.random.rand(N_DATA) * T
ud = exact(xd, td) + NOISE * np.random.randn(N_DATA)

xd = torch.tensor(xd, dtype=torch.float32, device=device).reshape(-1, 1)
td = torch.tensor(td, dtype=torch.float32, device=device).reshape(-1, 1)
ud = torch.tensor(ud, dtype=torch.float32, device=device).reshape(-1, 1)
print(f'{N_DATA} noisy measurements (true c={C_TRUE}, nu={NU_TRUE}).')

In [ ]:
# Cell 3 -- PINN with TWO trainable physical parameters
class InversePINN(nn.Module):
    def __init__(self, h=32, c_init=0.3, nu_init=0.1):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, h), nn.Tanh(),
                                 nn.Linear(h, 1))
        self.c      = nn.Parameter(torch.tensor(c_init))
        # learn log(nu) so nu = exp(log_nu) stays strictly positive (no backward diffusion)
        self.log_nu = nn.Parameter(torch.log(torch.tensor(nu_init)))
    @property
    def nu(self):
        return torch.exp(self.log_nu)
    def forward(self, x, t):
        return self.net(torch.cat([x, t], 1))

model = InversePINN(c_init=0.3, nu_init=0.1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=5e-3)
mse = nn.MSELoss()

c_hist, nu_hist = [], []
t0 = time.perf_counter()
for e in range(5000):
    opt.zero_grad()
    xi = (torch.rand(2000, 1, device=device) * L).requires_grad_(True)
    ti = (torch.rand(2000, 1, device=device) * T).requires_grad_(True)
    ui  = model(xi, ti)
    u_x = torch.autograd.grad(ui,  xi, torch.ones_like(ui),  create_graph=True)[0]
    u_t = torch.autograd.grad(ui,  ti, torch.ones_like(ui),  create_graph=True)[0]
    u_xx= torch.autograd.grad(u_x, xi, torch.ones_like(u_x), create_graph=True)[0]
    res = u_t + model.c * u_x - model.nu * u_xx        # convection-diffusion residual
    loss = mse(res, torch.zeros_like(res)) + 10.0 * mse(model(xd, td), ud)
    loss.backward(); opt.step()

    c_hist.append(model.c.item()); nu_hist.append(model.nu.item())
    if e % 500 == 0:
        print(f'epoch {e:4d}  loss {loss.item():.2e}  c={model.c.item():.4f}  nu={model.nu.item():.4f}')
if device.type == 'cuda':
    torch.cuda.synchronize()
train_time = time.perf_counter() - t0

print(f'\nTraining time : {train_time:.3f} s')
print(f'c  : recovered {model.c.item():.4f}  |  true {C_TRUE}   ({abs(model.c.item()-C_TRUE)/C_TRUE*100:.2f}% err)')
print(f'nu : recovered {model.nu.item():.4f}  |  true {NU_TRUE}  ({abs(model.nu.item()-NU_TRUE)/NU_TRUE*100:.2f}% err)')

In [ ]:
# Cell 4 -- Both parameters converge; field is reconstructed
fig, ax = plt.subplots(1, 3, figsize=(16, 4.3))

ax[0].plot(c_hist, 'b'); ax[0].axhline(C_TRUE, color='g', ls='--', label=f'true={C_TRUE}')
ax[0].set_title('convection speed c'); ax[0].set_xlabel('epoch'); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(nu_hist, 'm'); ax[1].axhline(NU_TRUE, color='g', ls='--', label=f'true={NU_TRUE}')
ax[1].set_title('diffusion coefficient nu'); ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3)

x = np.linspace(0, L, 200)
with torch.no_grad():
    xe = torch.tensor(x, dtype=torch.float32, device=device).reshape(-1, 1)
    u_pred = model(xe, torch.full_like(xe, T)).cpu().numpy().ravel()
ax[2].plot(x, exact(x, 0.0), 'k:', label='initial (t=0)')
ax[2].plot(x, exact(x, T),   'g',  lw=2.5, label='exact (t=T)')
ax[2].plot(x, u_pred,        'r-.', label='PINN (t=T)')
ax[2].set_title('field: advected + diffused'); ax[2].set_xlabel('x'); ax[2].legend(); ax[2].grid(alpha=.3)
plt.tight_layout(); plt.show()

## Takeaways
- One training run recovers **both** $c$ and $\nu$ from ~80 noisy points, and
  reconstructs the field (notice the pulse both **moves** and **flattens/widens**).
- Two physical effects are disentangled from the same data: the mean shift fixes
  $c$, the spreading fixes $\nu$. Autograd supplies $u_x, u_t, u_{xx}$ and the
  gradients w.r.t. the unknowns for free.
- With a classical solver this is a 2-parameter optimization wrapping **many full
  forward simulations**, plus adjoint/sensitivity machinery for the gradients.
- Learning $\log\nu$ (instead of $\nu$) keeps diffusion positive and stable.

**Try it:** change `C_TRUE`/`NU_TRUE`, raise `NOISE`, or cut `N_DATA` and watch how
the two recovered parameters degrade.